In [1]:
!pip install kaggle

In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"tanushakatakam","key":"4d681e49ee2e638d6a6cea0b418d3d8a"}'}

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
!kaggle datasets download blastchar/telco-customer-churn

Dataset URL: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
License(s): copyright-authors
100% 172k/172k [00:00<00:00, 115MB/s]



In [5]:
!unzip -o telco-customer-churn.zip -d telco-customer-churn

Archive:  telco-customer-churn.zip
  inflating: telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv  


In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression

# **Load Dataset**

In [7]:
df = pd.read_csv("telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.head())
print(df.info())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [8]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# **Data Cleaning**

In [9]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(df.isnull().sum())
df.dropna(subset=['TotalCharges'],inplace=True)
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})
df.head()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


Save original columns, because later get_dummies() will remove these columns.

In [10]:
original_contract = df['Contract']
original_payment = df['PaymentMethod']

# **Feature** **Engineering**

In [11]:
df.drop('customerID', axis=1, inplace=True)
df = pd.get_dummies(df, drop_first=True)

Restore original labels, for tableau as humans understand "One year" not dummy varaibles.

In [12]:
df['Contract_Type'] = original_contract
df['Payment_Method_Type'] = original_payment

# **Train Test Split**

In [13]:
X = df.drop(['Churn', 'Contract_Type', 'Payment_Method_Type'] , axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# **Model Training**

In [14]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(class_weight='balanced', max_iter=1000)

# **Model Evaluation**

In [29]:
y_pred = lr.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.91      0.71      0.80      1033
           1       0.50      0.80      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.71      1407
weighted avg       0.80      0.73      0.75      1407



In [31]:
print("Test Accuracy:", accuracy_score(y_test, y_pred))

Test Accuracy: 0.7334754797441365


In [34]:
y_pred = lr.predict(X_train)
print("Train Accuracy:", accuracy_score(y_train, y_pred))

Train Accuracy: 0.7550222222222223


# **Predictions**

In [16]:
df['Predicted_Churn'] = lr.predict(X)
df['Churn_Probability'] = lr.predict_proba(X)[:,1]

# **Risk Segmentation**

In [17]:
def risk(x):
    if x > 0.6:
        return "High"
    elif x > 0.3:
        return "Medium"
    else:
        return "Low"

df['Risk_Level'] = df['Churn_Probability'].apply(risk)

# **Export** **Data**

In [18]:
df.to_csv("churn_dashboard_data.csv", index=False)

Dataset with predictions

In [19]:
df.head(10)

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Contract_Type,Payment_Method_Type,Predicted_Churn,Churn_Probability,Risk_Level
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,True,False,True,False,Month-to-month,Electronic check,1,0.820877,High
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,One year,Mailed check,0,0.106111,Low
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,True,False,False,True,Month-to-month,Mailed check,1,0.513825,Medium
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,False,One year,Bank transfer (automatic),0,0.074239,Low
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,True,False,True,False,Month-to-month,Electronic check,1,0.854530,High
5,0,8,99.65,820.50,1,False,False,False,True,False,...,False,True,False,True,False,Month-to-month,Electronic check,1,0.905039,High
6,0,22,89.10,1949.40,0,True,False,True,True,False,...,False,True,True,False,False,Month-to-month,Credit card (automatic),1,0.703019,High
7,0,10,29.75,301.90,0,False,False,False,False,True,...,False,False,False,False,True,Month-to-month,Mailed check,1,0.528740,Medium
8,0,28,104.80,3046.05,1,False,True,False,True,False,...,False,True,False,True,False,Month-to-month,Electronic check,1,0.809870,High
9,0,62,56.15,3487.95,0,True,False,True,True,False,...,False,False,False,False,False,One year,Bank transfer (automatic),0,0.030110,Low


Download dataset with added predictions and load it on tableau

In [20]:
from google.colab import files
files.download("churn_dashboard_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>